In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 데이터셋 준비 (Iris 데이터를 PyTorch Tensor로 변환)
iris = load_iris()
X_raw = iris.data[:, 2:]   # 꽃잎 길이(Petal Length), 꽃잎 너비(Petal Width)
y_raw = iris.target        # 0, 1, 2 (setosa, versicolor, virginica)

# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# 피처 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Tensor 변환 (CrossEntropyLoss는 클래스 인덱스를 그대로 전달받음)
X = torch.tensor(X_train_scaled, dtype=torch.float32)
y = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)


# 2. 신경망 모델 정의 (nn.Module 상속)
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)   # 입력층 -> 은닉층
        self.sigmoid = nn.Sigmoid()                      # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size)   # 은닉층 -> 출력층

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 nn.CrossEntropyLoss 내부에서 처리되므로 생략
        return out


# 3. 모델, 손실 함수, 옵티마이저 생성
input_size = 2      # Petal Length, Petal Width
hidden_size = 5
output_size = 3      # setosa, versicolor, virginica
learning_rate = 0.5

torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


# 4. 모델 학습 루프 (Training Loop)
print("=== PyTorch 학습 시작 (Iris) ===")
epochs = 3000

for epoch in range(epochs):
    outputs = model(X)
    loss = criterion(outputs, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")


# 5. 테스트 샘플 예측
print("\n=== 테스트 샘플 예측 ===")
model.eval()  # 평가 모드 전환

# Iris 원본 스케일(cm) 기준으로 대표 샘플 직접 구성
# setosa, versicolor, virginica 순서로 petal length/width 값을 하나씩 선정
test_sample_raw = torch.tensor([
    [1.4, 0.2],   # Class 0 (setosa)
    [4.5, 1.5],   # Class 1 (versicolor)
    [5.8, 2.0],   # Class 2 (virginica)
], dtype=torch.float32)

# 학습 때 사용한 것과 동일한 scaler로 변환해야 함
test_sample_scaled = scaler.transform(test_sample_raw)
test_sample = torch.tensor(test_sample_scaled, dtype=torch.float32)

with torch.no_grad():  # 테스트 단계에서는 기울기 계산 불필요
    logits = model(test_sample)
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1} 확률 분포: {prob_list} -> 최종 예측 클래스: Class {pred.item()}")

=== PyTorch 학습 시작 (Iris) ===
Epoch  500 | Loss: 0.1088 | Accuracy: 95.0%
Epoch 1000 | Loss: 0.0910 | Accuracy: 95.0%
Epoch 1500 | Loss: 0.0871 | Accuracy: 95.0%
Epoch 2000 | Loss: 0.0856 | Accuracy: 95.0%
Epoch 2500 | Loss: 0.0849 | Accuracy: 95.0%
Epoch 3000 | Loss: 0.0844 | Accuracy: 95.0%

=== 테스트 샘플 예측 ===
샘플 1 확률 분포: [0.998, 0.002, 0.0] -> 최종 예측 클래스: Class 0
샘플 2 확률 분포: [0.0, 0.965, 0.035] -> 최종 예측 클래스: Class 1
샘플 3 확률 분포: [0.0, 0.004, 0.996] -> 최종 예측 클래스: Class 2


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# 데이터셋 로드 및 분할
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)

# 1. 데이터셋 준비 (NumPy 데이터를 PyTorch Tensor로 변환)
X = torch.tensor(X_train, dtype=torch.float32)
y = torch.tensor(y_train, dtype=torch.long)


# 2. 신경망 모델 정의
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  
        self.sigmoid = nn.Sigmoid()                    
        self.fc2 = nn.Linear(hidden_size, output_size) 

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  
        return out

# 3. 모델, 손실 함수, 옵티마이저 생성
input_size = 4  # Iris 특성 개수인 4로 수정
hidden_size = 8  # 입력이 늘어난 만큼 은닉층 노드도 조금 늘려주면 좋습니다
output_size = 3  # 세 가지 붓꽃 품종 (0, 1, 2)
learning_rate = 0.1 # 안정적인 학습을 위해 학습률을 0.1로 조정

torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


# 4. 모델 학습 루프
print("=== PyTorch 학습 시작 ===")
model.train()
epochs = 3000

for epoch in range(epochs):
    outputs = model(X)
    loss = criterion(outputs, y)

    optimizer.zero_grad()  
    loss.backward()        
    optimizer.step()       

    if (epoch + 1) % 500 == 0:
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")


# 5. 테스트 샘플 예측
print("\n=== 테스트 샘플 예측 ===")
model.eval() 
test_sample = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad(): 
    logits = model(test_sample)
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

# 테스트 데이터 정답률 확인을 위해 실제 정답(y_test)도 Tensor로 변환
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
test_accuracy = (predictions == y_test_tensor).float().mean() * 100
print(f"최종 테스트 데이터 정확도: {test_accuracy.item():.1f}%\n")

# 일부 샘플 출력 
for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1} 확률 분포: {prob_list} -> 예측: Class {pred.item()} (실제: Class {y_test[i]})")


=== PyTorch 학습 시작 ===
Epoch  500 | Loss: 0.3163 | Accuracy: 98.3%
Epoch 1000 | Loss: 0.1471 | Accuracy: 98.3%
Epoch 1500 | Loss: 0.1029 | Accuracy: 98.3%
Epoch 2000 | Loss: 0.0858 | Accuracy: 98.3%
Epoch 2500 | Loss: 0.0772 | Accuracy: 98.3%
Epoch 3000 | Loss: 0.0722 | Accuracy: 98.3%

=== 테스트 샘플 예측 ===
최종 테스트 데이터 정확도: 100.0%

샘플 1 확률 분포: [0.005, 0.969, 0.027] -> 예측: Class 1 (실제: Class 1)
샘플 2 확률 분포: [0.993, 0.007, 0.0] -> 예측: Class 0 (실제: Class 0)
샘플 3 확률 분포: [0.0, 0.004, 0.996] -> 예측: Class 2 (실제: Class 2)
샘플 4 확률 분포: [0.005, 0.957, 0.038] -> 예측: Class 1 (실제: Class 1)
샘플 5 확률 분포: [0.005, 0.981, 0.014] -> 예측: Class 1 (실제: Class 1)
샘플 6 확률 분포: [0.993, 0.007, 0.0] -> 예측: Class 0 (실제: Class 0)
샘플 7 확률 분포: [0.016, 0.983, 0.001] -> 예측: Class 1 (실제: Class 1)
샘플 8 확률 분포: [0.001, 0.184, 0.815] -> 예측: Class 2 (실제: Class 2)
샘플 9 확률 분포: [0.002, 0.57, 0.428] -> 예측: Class 1 (실제: Class 1)
샘플 10 확률 분포: [0.009, 0.989, 0.002] -> 예측: Class 1 (실제: Class 1)
샘플 11 확률 분포: [0.001, 0.353, 0.646] -> 예측: Class